In [1]:
tabla='hmcam10'
dim='dim_hoscamas'

In [2]:
import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()

In [3]:


oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM hmcam10", con=connection2)

connection2.close()




In [4]:
base2

,oricenasicod,cenasicod,arehoscod,servhoscod,estenfcod,habcod,camcod,tipcamcod,estcamcod,camobsdes,camestregcod,camflgfun
0,1,265,03,AC1,02,0004,17,1,5,,0,0
1,1,076,03,D11,04,01,01,2,6,,0,0
2,1,076,03,D11,04,03,A,2,6,,0,0
3,1,076,03,D11,04,03,D,2,6,,0,0
4,1,076,03,D11,04,04,B,2,6,,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
67195,1,002,03,H13,TH,GEN1,969,1,2,,1,0
67196,1,002,03,H13,TH,GEN1,977,1,2,,1,0
67197,1,002,03,H13,TH,AISL,979,1,1,,0,0
67198,1,002,03,H13,TH,GEN2,981,1,2,,1,0


In [5]:
base2.columns

Index(['oricenasicod', 'cenasicod', 'arehoscod', 'servhoscod', 'estenfcod',
       'habcod', 'camcod', 'tipcamcod', 'estcamcod', 'camobsdes',
       'camestregcod', 'camflgfun'],
      dtype='object')

In [6]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()


query_count_before = "SELECT COUNT(*) FROM hmcam10"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla hmcam10 antes de la inserción: {cant_antes}")


#connection3.execute('CREATE TEMPORARY TABLE tmp_hmcam10 ()')
base2.to_sql(name='tmp_hmcam10', con=engine3, if_exists='replace', index=False)

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL hmcam10 FALSO CON LO OBTENIDO DEL ORACLE

query="""

ALTER TABLE tmp_hmcam10 
ALTER COLUMN oricenasicod  TYPE character(1),
ALTER COLUMN cenasicod  TYPE character(3),
ALTER COLUMN arehoscod  TYPE character(2),
ALTER COLUMN servhoscod  TYPE character(3),
ALTER COLUMN estenfcod  TYPE character(2),
ALTER COLUMN habcod  TYPE character(4),
ALTER COLUMN camcod  TYPE character(5),
ALTER COLUMN tipcamcod  TYPE character(1),
ALTER COLUMN estcamcod  TYPE character(1),
ALTER COLUMN camobsdes  TYPE character(100),
ALTER COLUMN camestregcod  TYPE character(1),
ALTER COLUMN camflgfun  TYPE character(1);




UPDATE hmcam10 
SET 
tipcamcod = t.tipcamcod,
estcamcod = t.estcamcod,
camobsdes = t.camobsdes,
camestregcod = t.camestregcod,
camflgfun = t.camflgfun

FROM tmp_hmcam10 t 
WHERE hmcam10.oricenasicod = t.oricenasicod AND TRIM(hmcam10.oricenasicod) <> '' AND hmcam10.oricenasicod IS NOT NULL AND
hmcam10.cenasicod = t.cenasicod AND TRIM(hmcam10.cenasicod) <> '' AND hmcam10.cenasicod IS NOT NULL AND
hmcam10.arehoscod = t.arehoscod AND TRIM(hmcam10.arehoscod) <> '' AND hmcam10.arehoscod IS NOT NULL AND
hmcam10.servhoscod = t.servhoscod AND TRIM(hmcam10.servhoscod) <> '' AND hmcam10.servhoscod IS NOT NULL AND
hmcam10.estenfcod = t.estenfcod AND TRIM(hmcam10.estenfcod) <> '' AND hmcam10.estenfcod IS NOT NULL AND
hmcam10.habcod = t.habcod AND TRIM(hmcam10.habcod) <> '' AND hmcam10.habcod IS NOT NULL AND
hmcam10.camcod = t.camcod AND TRIM(hmcam10.camcod) <> '' AND hmcam10.camcod IS NOT NULL;


INSERT INTO hmcam10 
(oricenasicod,cenasicod,arehoscod,servhoscod,estenfcod,habcod,camcod,tipcamcod,estcamcod,camobsdes,camestregcod,camflgfun) 
SELECT 
oricenasicod,cenasicod,arehoscod,servhoscod,estenfcod,habcod,camcod,tipcamcod,estcamcod,camobsdes,camestregcod,camflgfun

FROM tmp_hmcam10  t
WHERE NOT EXISTS (
    SELECT 1 
    FROM hmcam10 
    WHERE hmcam10.oricenasicod = t.oricenasicod and hmcam10.oricenasicod IS NOT NULL AND
    hmcam10.cenasicod = t.cenasicod and hmcam10.cenasicod IS NOT NULL AND
    hmcam10.arehoscod = t.arehoscod and hmcam10.arehoscod IS NOT NULL AND
    hmcam10.servhoscod = t.servhoscod and hmcam10.servhoscod IS NOT NULL AND
    hmcam10.estenfcod = t.estenfcod and hmcam10.estenfcod IS NOT NULL AND
    hmcam10.habcod = t.habcod and hmcam10.habcod IS NOT NULL AND
    hmcam10.camcod = t.camcod and hmcam10.camcod IS NOT NULL
) ;


"""

c1= text(query)
connection3.execute(c1)

#BORRAMOS LAS TABLAS
query2="""
DROP TABLE tmp_hmcam10;
SELECT COUNT(*) FROM hmcam10;
"""


c2= text(query2)
cursor=connection3.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla mbtae10 después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")
base2 = pd.read_sql_query(f"SELECT * FROM hmcam10", con=connection3)


connection3.close()


Cantidad de filas en la tabla hmcam10 antes de la inserción: 66258
Cantidad de filas en la tabla mbtae10 después de la inserción: 67200
La cantidad de filas insertadas fue de: 942


In [7]:
#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT
base2.to_sql(name=f'tmp_{tabla}', con=engine4, if_exists='replace', index=False)

#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS
query_count_before = f"SELECT COUNT(*) FROM {dim}"
cant_antes = connection4.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} antes de la inserción: {cant_antes}")

query=f"""

ALTER TABLE tmp_{tabla} 
ALTER COLUMN oricenasicod  TYPE character(1),
ALTER COLUMN cenasicod  TYPE character(3),
ALTER COLUMN arehoscod  TYPE character(2),
ALTER COLUMN servhoscod  TYPE character(3),
ALTER COLUMN estenfcod  TYPE character(2),
ALTER COLUMN habcod  TYPE character(4),
ALTER COLUMN camcod  TYPE character(5),
ALTER COLUMN tipcamcod  TYPE character(1),
ALTER COLUMN estcamcod  TYPE character(1),
ALTER COLUMN camobsdes  TYPE character(100),
ALTER COLUMN camestregcod  TYPE character(1),
ALTER COLUMN camflgfun  TYPE character(1);


INSERT INTO {dim} 

(ori_cas,cas_cod,are_cod,ser_cod,enf_cod,hab_cod,cam_cod,tip_cam,est_cam,cam_obs,est_reg,cam_flg) 

SELECT 

oricenasicod,cenasicod,arehoscod,servhoscod,estenfcod,habcod,camcod,tipcamcod,estcamcod,camobsdes,camestregcod,camflgfun

FROM tmp_{tabla} t
WHERE NOT EXISTS (
    SELECT 1 
    FROM {dim} d 
    WHERE 
    
    d.ori_cas = t.oricenasicod AND
    d.cas_cod = t.cenasicod AND
    d.are_cod = t.arehoscod AND
    d.ser_cod = t.servhoscod AND
    d.enf_cod = t.estenfcod AND
    d.hab_cod = t.habcod AND
    d.cam_cod = t.camcod
);
"""



c1= text(query)
cursor=connection4.execute(c1)

#BORRAMOS LAS TABLAS
query2=f"""
DROP TABLE tmp_{tabla};
SELECT COUNT(*) FROM {dim};
"""

c2= text(query2)
cursor=connection4.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

Cantidad de filas en la tabla dim_hoscamas antes de la inserción: 66258
Cantidad de filas en la tabla dim_hoscamas después de la inserción: 67200
La cantidad de filas insertadas fue de: 942
